# 07. Model Explainability

This notebook explains the 90-day inactivity prediction model so the capstone project can connect model behavior to practical customer retention decisions.

**Primary inputs**

- `data/processed/customer_modeling_90d.csv`
- `data/processed/model_results_90d.csv`

**Primary outputs**

- `data/processed/feature_importance_90d.csv`
- `data/processed/shap_global_importance_90d.csv`

**Rerun safety**

`SAVE_OUTPUTS` is set to `False` by default. Normal reruns will not overwrite saved outputs. If you intentionally set `SAVE_OUTPUTS = True`, the save helper still compares the generated CSV content to the existing file and skips the write when nothing changed.

## 1. Safety Controls

In [ ]:
SAVE_OUTPUTS = False
RANDOM_STATE = 42
TEST_SIZE = 0.25
EXPLAINED_MODEL_NAME = "Gradient Boosting"
MAX_SHAP_ROWS = 1000

print("SAVE_OUTPUTS:", SAVE_OUTPUTS)
print("Explained model:", EXPLAINED_MODEL_NAME)

## 2. Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_PLOTS = True
    sns.set_theme(style="whitegrid", context="notebook")
except ImportError:
    HAS_PLOTS = False
    print("Plotting libraries are not available. Chart cells will be skipped.")

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    shap = None
    print("SHAP is not installed. SHAP tables and charts will be skipped.")

## 3. Project Paths

In [ ]:
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else Path(
    r"C:/Learning/BANA8083/AI-retention-decision-support"
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELING_PATH = PROCESSED_DIR / "customer_modeling_90d.csv"
MODEL_RESULTS_PATH = PROCESSED_DIR / "model_results_90d.csv"
FEATURE_IMPORTANCE_PATH = PROCESSED_DIR / "feature_importance_90d.csv"
SHAP_IMPORTANCE_PATH = PROCESSED_DIR / "shap_global_importance_90d.csv"

paths = pd.DataFrame(
    {
        "asset": [
            "modeling dataset",
            "model results",
            "feature importance output",
            "SHAP global importance output",
        ],
        "path": [
            MODELING_PATH,
            MODEL_RESULTS_PATH,
            FEATURE_IMPORTANCE_PATH,
            SHAP_IMPORTANCE_PATH,
        ],
        "exists": [
            MODELING_PATH.exists(),
            MODEL_RESULTS_PATH.exists(),
            FEATURE_IMPORTANCE_PATH.exists(),
            SHAP_IMPORTANCE_PATH.exists(),
        ],
    }
)
paths

## 4. Load Data

In [ ]:
if not MODELING_PATH.exists():
    raise FileNotFoundError(f"Missing modeling dataset: {MODELING_PATH}")

customer_model = pd.read_csv(
    MODELING_PATH,
    parse_dates=["first_purchase_date", "last_purchase_date"],
)

print(f"Rows: {len(customer_model):,}")
print(f"Columns: {customer_model.shape[1]:,}")
customer_model.head()

In [ ]:
required_columns = [
    "CustomerID",
    "first_purchase_date",
    "last_purchase_date",
    "primary_country",
    "active_90d",
    "inactive_90d",
]

missing_columns = [col for col in required_columns if col not in customer_model.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

data_quality = pd.DataFrame(
    {
        "check": [
            "rows",
            "unique_customers",
            "duplicate_customer_rows",
            "missing_target",
            "inactive_rate",
        ],
        "value": [
            len(customer_model),
            customer_model["CustomerID"].nunique(),
            customer_model["CustomerID"].duplicated().sum(),
            customer_model["inactive_90d"].isna().sum(),
            customer_model["inactive_90d"].mean(),
        ],
    }
)
data_quality

## 5. Feature Matrix and Train/Test Split

In [ ]:
TARGET = "inactive_90d"

drop_cols = [
    "CustomerID",
    "first_purchase_date",
    "last_purchase_date",
    "active_90d",
    "inactive_90d",
]

X = customer_model.drop(columns=drop_cols)
y = customer_model[TARGET]

categorical_cols = ["primary_country"]
numeric_cols = [col for col in X.columns if col not in categorical_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.DataFrame(
    {
        "split": ["train", "test"],
        "rows": [len(X_train), len(X_test)],
        "inactive_rate": [y_train.mean(), y_test.mean()],
    }
)
split_summary

## 6. Preprocess and Train Explained Model

In [ ]:
try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", onehot),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

classifier = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=RANDOM_STATE,
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", classifier),
    ]
)

model.fit(X_train, y_train)

y_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

print("Explained model:", EXPLAINED_MODEL_NAME)
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))
print("Classification report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
if MODEL_RESULTS_PATH.exists():
    saved_model_results = pd.read_csv(MODEL_RESULTS_PATH)
    saved_model_results
else:
    saved_model_results = pd.DataFrame()
    print("No saved model_results_90d.csv found for comparison.")

## 7. Feature Names After Preprocessing

In [ ]:
def clean_feature_name(feature_name):
    return (
        feature_name
        .replace("num__", "")
        .replace("cat__primary_country_", "primary_country=")
        .replace("cat__", "")
    )


feature_names = [clean_feature_name(name) for name in model.named_steps["preprocessor"].get_feature_names_out()]

feature_name_check = pd.DataFrame(
    {
        "feature_position": range(len(feature_names)),
        "feature": feature_names,
    }
)
feature_name_check.head(15)

## 8. Model Feature Importance

In [ ]:
classifier_step = model.named_steps["classifier"]

if not hasattr(classifier_step, "feature_importances_"):
    raise AttributeError(f"{EXPLAINED_MODEL_NAME} does not expose feature_importances_.")

feature_importance = (
    pd.DataFrame(
        {
            "feature": feature_names,
            "importance": classifier_step.feature_importances_,
        }
    )
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

feature_importance.head(15)

In [ ]:
if HAS_PLOTS:
    top_features = feature_importance.head(15).sort_values("importance", ascending=True)

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(data=top_features, x="importance", y="feature", color="#4C78A8", ax=ax)
    ax.set_title("Top Model Feature Importances")
    ax.set_xlabel("Importance")
    ax.set_ylabel("")
    sns.despine(left=True, bottom=True)
    plt.tight_layout()
    plt.show()
else:
    print("Skipping chart because plotting libraries are not available.")

## 9. SHAP Global Importance

In [ ]:
if SHAP_AVAILABLE:
    X_train_prepared = model.named_steps["preprocessor"].transform(X_train)
    X_test_prepared = model.named_steps["preprocessor"].transform(X_test)

    shap_row_count = min(MAX_SHAP_ROWS, X_test_prepared.shape[0])
    X_test_shap = X_test_prepared[:shap_row_count]

    explainer = shap.TreeExplainer(model.named_steps["classifier"])
    shap_values = explainer.shap_values(X_test_shap)

    if isinstance(shap_values, list):
        shap_values_for_class = shap_values[1]
    else:
        shap_values_for_class = shap_values

    shap_global_importance = (
        pd.DataFrame(
            {
                "feature": feature_names,
                "mean_abs_shap": np.abs(shap_values_for_class).mean(axis=0),
            }
        )
        .sort_values("mean_abs_shap", ascending=False)
        .reset_index(drop=True)
    )

    shap_global_importance.head(15)
else:
    shap_global_importance = pd.DataFrame(columns=["feature", "mean_abs_shap"])
    print("Skipping SHAP because the shap package is not installed.")

In [ ]:
if HAS_PLOTS and SHAP_AVAILABLE and not shap_global_importance.empty:
    top_shap = shap_global_importance.head(15).sort_values("mean_abs_shap", ascending=True)

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(data=top_shap, x="mean_abs_shap", y="feature", color="#F58518", ax=ax)
    ax.set_title("Top Global SHAP Importances")
    ax.set_xlabel("Mean Absolute SHAP Value")
    ax.set_ylabel("")
    sns.despine(left=True, bottom=True)
    plt.tight_layout()
    plt.show()
else:
    print("Skipping SHAP chart because plotting or SHAP is not available.")

## 10. Compare Importance Methods

In [ ]:
importance_comparison = feature_importance.merge(
    shap_global_importance,
    on="feature",
    how="outer",
)
importance_comparison["feature_importance_rank"] = importance_comparison["importance"].rank(
    ascending=False,
    method="min",
)
importance_comparison["shap_rank"] = importance_comparison["mean_abs_shap"].rank(
    ascending=False,
    method="min",
)
importance_comparison = importance_comparison.sort_values("feature_importance_rank").reset_index(drop=True)

importance_comparison.head(20)

## 11. Save Explainability Outputs Safely

In [ ]:
def save_csv_if_changed(dataframe, output_path):
    """Save a CSV only when saving is enabled and the file content changed."""
    if not SAVE_OUTPUTS:
        print(f"SAVE_OUTPUTS=False. Skipped save: {output_path}")
        return "skipped_save_outputs_false"

    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    new_csv = dataframe.to_csv(index=False)

    if output_path.exists():
        existing_csv = output_path.read_text(encoding="utf-8")
        if existing_csv == new_csv:
            print(f"No changes detected. Skipped save: {output_path}")
            return "unchanged"

    output_path.write_text(new_csv, encoding="utf-8")
    print(f"Saved updated file: {output_path}")
    return "updated"


save_status = {
    "feature_importance_90d": save_csv_if_changed(feature_importance, FEATURE_IMPORTANCE_PATH),
    "shap_global_importance_90d": save_csv_if_changed(shap_global_importance, SHAP_IMPORTANCE_PATH),
}

save_status

## 12. Explainability Takeaways

Use the generated tables and charts to connect the model back to the retention story:

- Features with high importance are the strongest signals used by the model.
- SHAP global importance provides a second view of which features most influence predictions.
- Recency, frequency, and monetary value should be reviewed closely because they are core behavioral retention signals.
- Explainability results should be used to support business recommendations, not as proof of causation.

The notebook is ready when it runs end to end and `SAVE_OUTPUTS` remains `False` unless you intentionally want to refresh saved explainability artifacts.